# PROBLEM 2: Character-level name generation (RNN)

This notebook walks through the assignment step by step: we load **1000 Indian names** from a text file, build a **character vocabulary**, train three small PyTorch models to predict the **next character**, then **sample** new names and measure **novelty** and **diversity**.

**TASK-0:** Names file (`TrainingNamesGPT.txt` — LLM-generated list).  
**TASK-1:** Vanilla RNN, BiLSTM (BLSTM), RNN + attention — architectures, parameter counts, hyperparameters.  
**TASK-2:** Novelty rate & diversity.  
**TASK-3:** Qualitative discussion + representative samples.  
**Extra:** Short written answers for the “which model is better?” and “vanilla RNN size” questions.

## TASK-0: Dataset

For this run we use **`TrainingNamesGPT.txt`** in the same folder as the notebook (one name per line, 1000 lines). That matches “generate names with an LLM and save them.”

The next code cell can **copy** that file to `TrainingNames.txt` if your rubric insists on that exact filename — otherwise you can ignore it and keep using the GPT file.

**Idea:** Each line is one full name (e.g. `Firstname Lastname`). We only `strip()` whitespace; later we lowercase everything for the character model.

In [1]:
# Optional: if your instructor wants the filename TrainingNames.txt exactly, duplicate the GPT file.
import shutil
from pathlib import Path

_base = Path.cwd()
_a, _b = _base / "TrainingNamesGPT.txt", _base / "TrainingNames.txt"
if not _b.is_file() and _a.is_file():
    shutil.copy(_a, _b)
    print("Wrote TrainingNames.txt from TrainingNamesGPT.txt")
elif _b.is_file():
    with _b.open(encoding="utf-8") as _f:
        _n = sum(1 for line in _f if line.strip())
    print(f"TrainingNames.txt already there ({_n} non-empty lines).")
elif _a.is_file():
    print("Using TrainingNamesGPT.txt only (see setup cell).")

TrainingNames.txt already there (1000 non-empty lines).


In [2]:
# Run once if needed:
%pip install -q torch pandas matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

BASE = Path.cwd()
# Prefer the LLM-generated list you saved; fall back to TrainingNames.txt if needed.
_gpt = BASE / "TrainingNamesGPT.txt"
_txt = BASE / "TrainingNames.txt"
if _gpt.is_file():
    NAMES_FILE = _gpt
elif _txt.is_file():
    NAMES_FILE = _txt
else:
    raise FileNotFoundError("Need TrainingNamesGPT.txt (or TrainingNames.txt) next to this notebook.")

print("Using names file:", NAMES_FILE.name)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

# --- hyperparameters (report these in your write-up) ---
EMB_DIM = 64
HIDDEN_SIZE = 128
NUM_LAYERS = 1
LR = 1e-3
BATCH_SIZE = 64
EPOCHS = 35
MAX_GEN_LEN = 40
N_GEN = 500  # how many names to sample for metrics

# load names
with NAMES_FILE.open(encoding="utf-8") as f:
    raw_names = [line.strip() for line in f if line.strip()]
print("loaded names:", len(raw_names))


Using names file: TrainingNamesGPT.txt
device: cpu
loaded names: 1000


In [4]:
# --- Character vocabulary ---
# We predict one character at a time. Special tokens: <pad> = 0 for batching, and "\n" = end of name.
PAD_IDX = 0


def build_vocab(names):
    """Collect every character that appears in any name (lowercased), plus newline as EOS."""
    chars = set()
    for n in names:
        # inner loop: walk each character in this name
        for c in n.lower():
            chars.add(c)
    chars.add("\n")  # end-of-sequence when generating
    chars = sorted(chars)
    c2i = {"<pad>": 0}
    # give each real char an id starting from 1 (0 stays padding)
    for i, c in enumerate(chars, start=1):
        c2i[c] = i
    i2c = {v: k for k, v in c2i.items()}
    return c2i, i2c


c2i, i2c = build_vocab(raw_names)
VOCAB_SIZE = len(c2i)
print("vocab size:", VOCAB_SIZE)


def encode(s: str):
    return [c2i[c] for c in s.lower()]


def decode(ids):
    """Turn predicted ids back into a string; stop at newline or pad."""
    out = []
    for i in ids:
        if i == 0:
            continue
        ch = i2c[i]
        if ch == "<pad>":
            continue
        if ch == "\n":
            break
        out.append(ch)
    return "".join(out)


class NameCharDataset(Dataset):
    """Next-character prediction: prefix -> next char (teacher forcing)."""

    def __init__(self, names, c2i):
        self.samples = []
        eos = c2i["\n"]
        for name in names:
            s = name.lower().strip() + "\n"
            ids = [c2i[c] for c in s]
            # for each position t: input = chars before t, target = char at t
            for t in range(len(ids)):
                prefix = ids[:t]
                target = ids[t]
                self.samples.append((prefix, target))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        prefix, target = self.samples[idx]
        return prefix, target


def collate_fn(batch):
    """Pad prefixes on the LEFT so the last column is the most recent character (RNN reads left→right)."""
    prefixes, targets = zip(*batch)
    lengths = torch.tensor([len(p) for p in prefixes], dtype=torch.long)
    max_len = int(lengths.max()) if len(lengths) else 1
    B = len(batch)
    x = torch.zeros(B, max_len, dtype=torch.long)  # PAD=0, left pad
    for i, p in enumerate(prefixes):
        if len(p) == 0:
            continue
        start = max_len - len(p)
        x[i, start:] = torch.tensor(p, dtype=torch.long)
    y = torch.tensor(targets, dtype=torch.long)
    return x, y, lengths


ds = NameCharDataset(raw_names, c2i)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
print("training samples (prefix,target pairs):", len(ds))


vocab size: 29
training samples (prefix,target pairs): 13336


## TASK-1: Models (what each one does)

### 1) Vanilla RNN
- **Embedding:** each character id → a vector of size `EMB_DIM`.
- **`nn.RNN`:** reads the padded prefix left to right; tanh recurrence. We take the hidden state at the **last time step** (rightmost column after left-padding).
- **`Linear`:** maps that hidden vector to logits over the vocabulary (next character).

### 2) BiLSTM (BLSTM)
- **Bidirectional LSTM:** for the same padded prefix, a forward LSTM and a backward LSTM both run; outputs are **concatenated** (size `2 * HIDDEN_SIZE`).
- Again we read the **last** time step and project to vocab. (Both directions only see the prefix so far — not “future” letters of the full name.)

### 3) RNN + basic attention
- **`nn.RNN`** gives a hidden vector at **every** position.
- **Attention:** use the last hidden as a **query**, compare to keys from all positions, **softmax** → weights, then **weighted sum** of RNN outputs = context.
- Concatenate **query** and **context**, then `Linear` → logits. Padding positions get a large negative score so attention ignores them.

In [5]:
def count_params(model):
    """Trainable scalars (weights + biases) — what the assignment asks for."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def model_size_mb(model):
    """Approximate memory if all tensors were float32 in RAM (good enough for the report)."""
    nbytes = sum(p.numel() * p.element_size() for p in model.parameters())
    return nbytes / (1024**2)


class VanillaCharRNN(nn.Module):
    """Smallest model: embed → RNN over time → linear classifier for next char."""

    def __init__(self, vocab_size, emb_dim, hidden_dim, num_layers):
        super().__init__()
        self.pad_idx = 0
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=self.pad_idx)
        self.rnn = nn.RNN(emb_dim, hidden_dim, num_layers, batch_first=True, nonlinearity="tanh")
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        e = self.emb(x)
        out, h = self.rnn(e)
        last = out[:, -1, :]  # last timestep = end of prefix
        return self.fc(last)


class BiLSTMChar(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_layers):
        super().__init__()
        self.pad_idx = 0
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=self.pad_idx)
        self.lstm = nn.LSTM(
            emb_dim,
            hidden_dim,
            num_layers,
            batch_first=True,
            bidirectional=True,
        )
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)

    def forward(self, x):
        e = self.emb(x)
        out, _ = self.lstm(e)
        last = out[:, -1, :]
        return self.fc(last)


class RNNAttentionChar(nn.Module):
    """Single-layer RNN + one-head dot-product attention over time steps."""

    def __init__(self, vocab_size, emb_dim, hidden_dim, num_layers):
        super().__init__()
        self.pad_idx = 0
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=self.pad_idx)
        self.rnn = nn.RNN(emb_dim, hidden_dim, num_layers, batch_first=True, nonlinearity="tanh")
        self.Wk = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)

    def forward(self, x):
        e = self.emb(x)
        out, h = self.rnn(e)
        query = out[:, -1, :]
        key = self.Wk(out)
        scores = torch.bmm(key, query.unsqueeze(-1)).squeeze(-1)
        scores = scores.masked_fill(x == 0, -1e9)
        attn = torch.softmax(scores, dim=1)
        ctx = torch.bmm(attn.unsqueeze(1), out).squeeze(1)
        cat = torch.cat([query, ctx], dim=-1)
        return self.fc(cat)


models = {
    "Vanilla RNN": VanillaCharRNN(VOCAB_SIZE, EMB_DIM, HIDDEN_SIZE, NUM_LAYERS),
    "BiLSTM": BiLSTMChar(VOCAB_SIZE, EMB_DIM, HIDDEN_SIZE, NUM_LAYERS),
    "RNN + Attention": RNNAttentionChar(VOCAB_SIZE, EMB_DIM, HIDDEN_SIZE, NUM_LAYERS),
}

for name, m in models.items():
    m.to(DEVICE)
    print(name, "trainable parameters:", count_params(m))

# P2 rubric: vanilla RNN parameter count + model size (MB)
_v = models["Vanilla RNN"]
print()
print("--- Vanilla RNN (for report) ---")
print("  trainable parameters:", count_params(_v))
print("  model size (parameters, ~MB):", round(model_size_mb(_v), 4), "MB")

print("\nHyperparameters: emb_dim=", EMB_DIM, " hidden=", HIDDEN_SIZE, " layers=", NUM_LAYERS, " lr=", LR, " epochs=", EPOCHS)


Vanilla RNN trainable parameters: 30429
BiLSTM trainable parameters: 207965
RNN + Attention trainable parameters: 50525

--- Vanilla RNN (for report) ---
  trainable parameters: 30429
  model size (parameters, ~MB): 0.1161 MB

Hyperparameters: emb_dim= 64  hidden= 128  layers= 1  lr= 0.001  epochs= 35


**TASK-1 summary (copy into your report)**

| Setting | Value |
|--------|-------|
| Embedding dim | 64 |
| Hidden size | 128 |
| RNN/LSTM layers | 1 |
| Learning rate | 0.001 |
| Optimizer | Adam |
| Batch size | 64 |
| Training epochs | 35 |

**Vanilla RNN:** after running the model cell, the notebook prints **trainable parameters** and **model size (~MB)** for the rubric.

BiLSTM and RNN+Attention counts print above too (BiLSTM is largest because of LSTM gates × bidirection).


In [6]:
def train_one_model(model, loader, epochs=EPOCHS):
    """Train with Adam + cross-entropy; return one average loss per epoch (for curves)."""
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss()
    history = []
    model.train()
    for ep in range(1, epochs + 1):
        total, n = 0.0, 0
        for x, y, lengths in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()
            opt.step()
            total += loss.item() * x.size(0)
            n += x.size(0)
        avg = total / max(n, 1)
        history.append(avg)
        if ep == 1 or ep % 5 == 0 or ep == epochs:
            print(f"  epoch {ep:3d}  loss {avg:.4f}")
    return history


loss_histories = {}
trained = {}
for name, model in models.items():
    print("\nTraining:", name)
    loss_histories[name] = train_one_model(model, loader)
    trained[name] = model

# TASK-2: combined training-loss figure for the report
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.5))
epoch_axis = list(range(1, EPOCHS + 1))
for name, hist in loss_histories.items():
    ax.plot(epoch_axis, hist, label=name, linewidth=2, marker="o", markersize=3)
ax.set_xlabel("Epoch")
ax.set_ylabel("Average cross-entropy loss")
ax.set_title("Problem 2 — training loss comparison")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.35)
plt.tight_layout()
_loss_plot_path = BASE / "problem2_loss_placeholder.png"
plt.savefig(_loss_plot_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved figure:", _loss_plot_path.resolve())



Training: Vanilla RNN
  epoch   1  loss 2.3389
  epoch   5  loss 1.3920
  epoch  10  loss 1.1330
  epoch  15  loss 1.0087
  epoch  20  loss 0.9291
  epoch  25  loss 0.8740
  epoch  30  loss 0.8284
  epoch  35  loss 0.7878

Training: BiLSTM
  epoch   1  loss 2.3875
  epoch   5  loss 1.3792
  epoch  10  loss 1.0850
  epoch  15  loss 0.9423
  epoch  20  loss 0.8502
  epoch  25  loss 0.7804
  epoch  30  loss 0.7234
  epoch  35  loss 0.6819

Training: RNN + Attention
  epoch   1  loss 2.2611
  epoch   5  loss 1.4022
  epoch  10  loss 1.1420
  epoch  15  loss 1.0059
  epoch  20  loss 0.9239
  epoch  25  loss 0.8675
  epoch  30  loss 0.8232
  epoch  35  loss 0.7706


In [7]:
@torch.no_grad()
def sample_name(model, max_len=MAX_GEN_LEN, temperature=0.85):
    """Autoregressive generation: repeatedly predict next char and append to prefix.
    Temperature > 1 flattens the softmax (more random); < 1 peaks (more greedy)."""
    model.eval()
    generated = []
    prefix = []
    eos = c2i["\n"]
    for _ in range(max_len):
        L = len(prefix)
        T = max(L, 1)
        x = torch.zeros(1, T, dtype=torch.long, device=DEVICE)
        if L > 0:
            x[0, -L:] = torch.tensor(prefix, dtype=torch.long, device=DEVICE)
        logits = model(x) / temperature
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, 1).item()
        if next_id == eos:
            break
        generated.append(next_id)
        prefix.append(next_id)
    return decode(generated)


train_set = set(n.lower() for n in raw_names)


def evaluate_model(model, n_samples=N_GEN):
    """TASK-2 metrics: novelty = fraction not in training set; diversity = unique / total."""
    gens = []
    for _ in range(n_samples):
        s = sample_name(model)
        if len(s) < 2:
            continue
        gens.append(s.strip())
    if not gens:
        return {"novelty": 0.0, "diversity": 0.0, "samples": []}
    novel = sum(1 for g in gens if g.lower() not in train_set)
    novelty = novel / len(gens)
    diversity = len(set(gens)) / len(gens)
    return {"novelty": novelty, "diversity": diversity, "samples": gens}


print("\n=== TASK-2: quantitative metrics ===")
rows = []
for name, model in trained.items():
    m = evaluate_model(model)
    rows.append((name, m["novelty"], m["diversity"]))
    print(f"{name:18s}  novelty={m['novelty']:.3f}  diversity={m['diversity']:.3f}")

import pandas as pd

df = pd.DataFrame(rows, columns=["model", "novelty_rate", "diversity"])
try:
    from IPython.display import display

    display(df)
except Exception:
    print(df.to_string())

# Bar comparison for TASK-2 (three models × two metrics; clearer than a literal histogram with 3 points)
import matplotlib.pyplot as plt
import numpy as np

_labels = df["model"].tolist()
_x = np.arange(len(_labels))
_w = 0.35
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.bar(_x - _w / 2, df["novelty_rate"], _w, label="novelty rate", color="#2E86AB")
ax.bar(_x + _w / 2, df["diversity"], _w, label="diversity", color="#A23B72")
ax.set_ylabel("Score")
ax.set_title("Problem 2 — novelty rate and diversity by model")
ax.set_xticks(_x)
ax.set_xticklabels(_labels, rotation=18, ha="right")
ax.set_ylim(0, 1.05)
ax.legend(loc="upper left")
ax.grid(True, axis="y", alpha=0.35)
plt.tight_layout()
_bar_path = BASE / "problem2_novelty_diversity_bars.png"
plt.savefig(_bar_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved figure:", _bar_path.resolve())



=== TASK-2: quantitative metrics ===
Vanilla RNN         novelty=0.998  diversity=0.558
BiLSTM              novelty=1.000  diversity=0.889
RNN + Attention     novelty=0.990  diversity=0.782


,model,novelty_rate,diversity
0,Vanilla RNN,0.997821,0.557734
1,BiLSTM,1.000000,0.888889
2,RNN + Attention,0.989980,0.781563


## Discussion (P2): Which model is “better” — RNN, BiLSTM, or RNN + Attention?

### Comparison (TASK-2 — use your printed table + loss figure)

After **35 epochs** on the same data, **BiLSTM** typically **wins on all three signals**: it reaches the **lowest training loss** (next-character cross-entropy), the **highest novelty rate** (generated names not in the training list), and the **highest diversity** (unique strings ÷ total samples). **RNN + Attention** usually **beats the vanilla RNN** on loss and diversity but stays **below BiLSTM**. The **vanilla RNN** is the **lightest model** and trains quickly, yet it often **underfits** a bit on this task, so its loss stays higher and sampling **repeats** similar strings more often (lower diversity).

**Why BiLSTM helps on a small name corpus:** names are short, but the model must remember **earlier letters** in the prefix (e.g. syllables, spaces). **LSTM gates** reduce vanishing-gradient issues compared to a plain **tanh RNN**, and the **backward LSTM** sees the whole prefix from the other side, which gives a richer summary before predicting the next character. That usually improves **fit** (loss) and makes **stochastic sampling** less repetitive (**diversity**), while **novelty** stays high because the model is not simply memorizing one line.

The **training cell** saves **`problem2_loss_placeholder.png`** in the project folder (next to the notebook) — embed that image in your report. The **model definition cell** prints **vanilla RNN parameter count** and **size (~MB)** for the rubric.


## TASK-3: Qualitative analysis

Read the printed names like a human:

- **Realism:** Do you see plausible “Firstname Lastname” patterns, reasonable length, mostly letters/spaces?
- **Failure modes:** too short (`ab`), weird spacing, repeated syllables, or strings that are basically a training name with one letter changed.

Below we print **20 unique samples per model** (sampling loop skips duplicates; still not guaranteed novel vs training).

In [8]:
# TASK-3: print a handful of generations per model for the report
print("=== Representative samples ===\n")
for name, model in trained.items():
    print("---", name, "---")
    seen = set()
    out = []
    tries = 0
    # resample until we have 20 different strings (generation is stochastic)
    while len(out) < 20 and tries < 200:
        tries += 1
        s = sample_name(model, temperature=0.9)
        if s and s not in seen:
            seen.add(s)
            out.append(s)
    for s in out:
        print(" ", s)
    print()


=== Representative samples ===

--- Vanilla RNN ---
  hanna
  lanan
  redey
  redha
  ghetam
  er
  bhandar
  bharma
  anni
  achord
  luk taik
  man
  ti
  ofi
  jandiy
  dhy
  ukh
  i
  hetti khatala
  purik gi

--- BiLSTM ---
  ga dwajoj choudhry
  dh kurya
  shul menon
  riy
  tej joshi
  awat
  hanrey ahmed
  arg yadav
  dha joshi
  y
  minen
  sha mehta
   meher naj
  dhy
  kabhra bharge
  i bhatia
  them ghosh
  vya choudhury
  kshant chatterjee
  hee

--- RNN + Attention ---
  ree reddy
  ni ram
  ur jali
  ill bhatia
  shi varma
  dev kapoor
  khel opnachary
  ya khan
  a dubey
  tej choudhary
  bhita puryap
  ali patil
  a kapoor
  udha dutta
  rohit patel
  ali sinha
  rita verma
  an khan
  bhata sundaram
  al oppoor

